<a href="https://colab.research.google.com/github/danilo-tmb/deep-learning/blob/main/01_setup_and_data_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 - Setup and data check (PressIQ)
Goal: connect to SkillCorner, find NWSL 2025 and FAWSL 2024/25, download ONE match, look at it.

**Before saving to GitHub:** Edit > Clear all outputs (outputs can contain data).

## 1. Install and import

In [ ]:
!pip install -U skillcorner -q
import json, os, io
import pandas as pd
from skillcorner.client import SkillcornerClient

## 2. Mount Google Drive (data lives here, never in GitHub)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/pressiq_data'
os.makedirs(f'{DATA_DIR}/tracking', exist_ok=True)
os.makedirs(f'{DATA_DIR}/events', exist_ok=True)
print('Data folder:', DATA_DIR)

## 3. Connect (uses Colab Secrets `SC_USER` and `SC_PASS`)

In [ ]:
from google.colab import userdata
client = SkillcornerClient(username=userdata.get('SC_USER'),
                           password=userdata.get('SC_PASS'))
print('Client created')

## 4. Find the competition editions you have access to
Look for NWSL 2025 and FAWSL 2024/25 and note their `id`.

In [ ]:
editions = client.get_competition_editions()
ed = pd.json_normalize(editions)
print(len(ed), 'editions')
print(ed.columns.tolist())   # check which columns exist
ed.head(50)

In [ ]:
w = ed[ed['competition.gender'] == 'female']
mask = (w['name'].str.contains('NWSL|National Women', case=False) & (w['season.name'] == '2025')) | \
       (w['name'].str.contains('Super League|WSL', case=False) & (w['season.name'] == '2024/2025'))
w[mask][['id', 'name', 'season.name']]

## 5. Fill in the two edition ids from step 4

In [ ]:
NWSL_ID  = 1093   # e.g. 1234
FAWSL_ID = 0   # e.g. 5678
assert NWSL_ID , 'Fill in both ids from step 4'

## 6. List matches and save the lists to Drive

In [ ]:
all_matches = []
for league, ed_id in [('NWSL', NWSL_ID)]:
    m = pd.json_normalize(client.get_matches(params={'competition_edition': ed_id}))
    m['league'] = league
    all_matches.append(m)
    print(league, len(m), 'matches')

matches = pd.concat(all_matches, ignore_index=True)
matches.to_csv(f'{DATA_DIR}/matches.csv', index=False)
print(matches.columns.tolist())
matches.head()

In [ ]:
w = ed[(ed['competition.gender'] == 'female') & ed['name'].str.contains('ENG', case=False)]
for _, r in w.iterrows():
    n = len(client.get_matches(params={'competition_edition': int(r['id'])}))
    if n > 0:
        print(int(r['id']), r['name'], '->', n, 'matches')
print('done')

## 7. Download tracking for ONE match (saved to Drive, skipped if already there)

In [ ]:
MATCH_ID = int(matches['id'].iloc[0])   # first match; change if you want
track_path = f'{DATA_DIR}/tracking/{MATCH_ID}.jsonl'

if not os.path.exists(track_path):
    client.save_match_tracking_data(match_id=MATCH_ID, filepath=track_path,
                                    params={'data_version': 3})
print('Saved:', track_path, round(os.path.getsize(track_path) / 1e6, 1), 'MB')

## 8. Look at the tracking data

In [ ]:
with open(track_path) as f:
    head = f.read(1)

if head == '[':                      # one big JSON list
    with open(track_path) as f:
        frames = json.load(f)
else:                                # one JSON object per line
    frames = [json.loads(l) for l in open(track_path) if l.strip()]

print('Frames:', len(frames), '| approx minutes at 10 Hz:', round(len(frames) / 600, 1))
print('Keys in one frame:', list(frames[1000].keys()))
frames[1000]

## 9. Download dynamic events for the same match
If this cell errors, paste the error to Claude.

In [ ]:
events = client.get_dynamic_events(match_id=MATCH_ID)
raw = events.decode('utf-8') if isinstance(events, bytes) else events

if isinstance(raw, (list, dict)):
    ev = pd.json_normalize(raw)
elif raw.lstrip().startswith(('[', '{')):
    ev = pd.json_normalize(json.loads(raw))
else:
    ev = pd.read_csv(io.StringIO(raw))

ev.to_csv(f'{DATA_DIR}/events/{MATCH_ID}.csv', index=False)
print(ev.shape)
print(ev.columns.tolist()[:40])

## 10. First look at pressing-related events

In [ ]:
for col in ['event_type', 'event_subtype']:
    if col in ev.columns:
        print(ev[col].value_counts(), '\n')

In [ ]:
#counting episodes in the match
obe = ev[ev['event_type'] == 'on_ball_engagement']
chains = obe[obe['pressing_chain'] == True]

print('Pressing chains in this match:', chains['pressing_chain_index'].nunique())
print(chains.groupby('pressing_chain_index')['pressing_chain_end_type']
            .first().value_counts(dropna=False))
print('Avg chain length:', chains.groupby('pressing_chain_index').size().mean().round(2))

## Findings (Nov 19, 2026)

**Data access**
- Only NWSL 2025 is available (edition id 1093, 189 matches). FAWSL is not provided (confirmed by organizers).
- Login: Python client with username + password (stored in Colab Secrets).

**Tracking data**
- 10 Hz, about 60,000 frames per match (about 101 min).
- Per frame: ball (x, y, z), possession team, 22 players (x, y, player_id, is_detected).
- Coordinates in metres, (0, 0) = centre spot.
- `is_detected = False` means the player is off camera and the position is estimated. This is a data-quality issue to track.
- No team label per player: map player_id to team with `client.get_match()`.
- File format: one JSON list (not JSONL).

**Dynamic events**
- One match: 4,099 events, 310 columns. Returned as CSV bytes.
- Event types: passing_option, on_ball_engagement, player_possession, off_ball_run.
- Pressing subtypes: pressing, pressure, recovery_press, counter_press, other.
- Linked to tracking via `frame_start` / `frame_end`.

**Pressing episodes**
- SkillCorner labels pressing chains (`pressing_chain_index`), with an outcome (`pressing_chain_end_type`: regain / disruption / None).
- One match: 80 chains, avg 3.55 actions per chain; outcomes 30 regain, 4 disruption, 46 none.
- Estimate: about 15,000 chains across 189 matches, so the sample size is enough for CNN + LSTM.

**Caveats**
- Based on one match only. Confirm on about 10 matches from different teams.
- Chains within a match are not independent, so split train/test by match.

**Next**
- Notebook 02: download about 10 matches, confirm episode counts, check Drive storage per match.